# 01 — Getting myself familiar with the datasets

Sources: [vaastav/Fantasy-Premier-League](https://github.com/vaastav/Fantasy-Premier-League) (match stats, seasons 2021-22 to 2025-26) and [transfermarkt-datasets](https://github.com/dcaribou/transfermarkt-datasets) (market values, until June 2026).

Loading code is in `src/load.py`; the cells below show why each cleaning step was needed.

**Data decisions**
- Seasons 2021-22 to 2025-26. Split: train 2021-22 to 2023-24, validation 2024-25, test 2025-26.
- *(xG, xA…) not used* : zeros before GW16 of 2022-23 and absent in 2021-22. 
- I noticed that threat and creativity, available in every season, are strongly correlated with xG/90 and xA/90.
- *Starts not used*: absent before 2022-23 (kind of replaced by minutes).
- *Seasons before 2021-22 not used.*
- *cleaned_merged_seasons.csv not used* : individual season files are more complete.
- *Excluded for leakage* : xP, value, selected, everything related to transfers (FPL), and market_value_in_eur , highest_market_value_in_eur, contract_expiration_date from Transfermarkt players (they describe the player in June 2026, not at the valuation date).

In [1]:
import sys
sys.path.append("..")  

import pandas as pd
from src.load import load_gw, load_valuations, load_players

V = "../data/vaastav/data/"  # raw files, to look at what load_gw cleans

gw = load_gw()
val = load_valuations()
players = load_players()
print(gw.shape, val.shape, players.shape)
gw.groupby("season").size()

(138717, 22) (11267, 6) (50149, 5)


season
2021-22    25447
2022-23    26505
2023-24    29725
2024-25    27283
2025-26    29757
dtype: int64

## Assistant managers in 2024-25

The raw 2024-25 file has a position AM: these are managers (FPL "assistant manager" chip), not players. 

In [2]:
raw_2425 = pd.read_csv(V + "2024-25/gws/merged_gw.csv")
am = raw_2425[raw_2425["position"] == "AM"]

print(len(am), "AM rows")
am["name"].unique()

322 AM rows


<StringArray>
[                       'Fabian Hürzeler',
                           'Thomas Frank',
       'Marco Alexandre Saraiva da Silva',
                          'Pep Guardiola',
   'Nuno Herlander Simões Espírito Santo',
 'Vítor Manuel de Oliveira Lopes Pereira',
                         'Kieran McKenna',
      'Ruben Filipe Marques Diogo Amorim',
                           'Mikel Arteta',
                   'Ruud van Nistelrooij',
                          'Andoni Iraola',
                         'Oliver Glasner',
                       'Ange Postecoglou',
                           'Enzo Maresca',
                          'Graham Potter',
                             'Ivan Juric',
                             'Eddie Howe',
                             'Unai Emery',
                            'David Moyes',
                              'Arne Slot',
                             'Simon Rusk']
Length: 21, dtype: str

## Goalkeeper label in 2021-22

2021-22 uses both GK and GKP. load_gw replace everything to GK.

In [3]:
raw_2122 = pd.read_csv(V + "2021-22/gws/merged_gw.csv")
raw_2122["position"].unique()

<StringArray>
['DEF', 'FWD', 'MID', 'GK', 'GKP']
Length: 5, dtype: str

## xG and xA are missing before GW16 of 2022-23

FPL only started publishing expected stats in November 2022. Before that, the values are 0 not NaN.

In [4]:
raw_2223 = pd.read_csv(V + "2022-23/gws/merged_gw.csv")
played = raw_2223[raw_2223["minutes"] > 0]

# total xG and xA per gameweek
played.groupby("GW")[["expected_goals", "expected_assists"]].sum().head(18)

,expected_goals,expected_assists
GW,,
1,0.0000,0.00000
2,0.0000,0.00000
3,0.0000,0.00000
4,0.0000,0.00000
5,0.0000,0.00000
6,0.0000,0.00000
8,0.0000,0.00000
9,0.0000,0.00000
10,0.0000,0.00000


## Can *threat* and *creativity* replace xG and xA?

Spearman correlation of per-90 stats, 2023-24, players with at least 900 minutes.

In [5]:
raw_2324 = pd.read_csv(V + "2023-24/gws/merged_gw.csv")

# one row per player for the whole season
totals = raw_2324.groupby("element")[["minutes", "expected_goals", "expected_assists", "threat", "creativity"]].sum()
totals = totals[totals["minutes"] >= 900]

# stats per 90 minutes
for col in ["expected_goals", "expected_assists", "threat", "creativity"]:
    totals[col + "_90"] = totals[col] / totals["minutes"] * 90

print(len(totals), "players")
print("threat vs xG:", totals["threat_90"].corr(totals["expected_goals_90"], method="spearman"))
print("creativity vs xA:", totals["creativity_90"].corr(totals["expected_assists_90"], method="spearman"))

339 players
threat vs xG: 0.9571319503455671
creativity vs xA: 0.9310743789549495


## Transfermarkt: the club is the club at the valuation date

Declan Rice moved from West Ham to Arsenal in July 2023: `current_club_name` follows the transfer, so it is safe to use for matching season by season.

In [6]:
rice_id = players[players["name"] == "Declan Rice"]["player_id"].iloc[0]

val[val["player_id"] == rice_id].sort_values("date")

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
29,357662,2021-07-15,70000000,West Ham United,379,GB1
939,357662,2021-12-23,75000000,West Ham United,379,GB1
2010,357662,2022-06-15,80000000,West Ham United,379,GB1
3409,357662,2022-11-03,80000000,West Ham United,379,GB1
5115,357662,2023-06-20,90000000,West Ham United,379,GB1
5509,357662,2023-10-09,100000000,Arsenal FC,11,GB1
6076,357662,2023-12-19,110000000,Arsenal FC,11,GB1
7246,357662,2024-05-27,120000000,Arsenal FC,11,GB1
7912,357662,2024-12-16,110000000,Arsenal FC,11,GB1
9031,357662,2025-05-30,120000000,Arsenal FC,11,GB1


## The `GB1` filter is not enough on its own

It keeps reserve teams nd lower-division clubs. I chose that the 20 Premier League clubs of each season will be kept during matching.

In [7]:
print(val["current_club_name"].nunique(), "club names in val")
val["current_club_name"].value_counts().tail(10)

351 club names in val


current_club_name
Shabab Al-Ahli Club                1
Al-Jazira (Abu Dhabi)              1
Union Titus Petange                1
FC Lausanne-Sport                  1
Bandirmaspor                       1
Hertha BSC                         1
Wydad Casablanca                   1
Al-Rayyan SC                       1
Inverness Caledonian Thistle FC    1
Paris Saint-Germain                1
Name: count, dtype: int64

## Mid-season transfers in FPL data

In vaastav, `team` is the club at the time of each match and `element` stays the same within a season. FPL names can be very different from Transfermarkt names (here "Jorginho").

In [8]:
jorginho = gw[(gw["season"] == "2022-23") & (gw["name"].str.contains("Frello"))]

print(jorginho["name"].unique(), jorginho["element"].unique())
jorginho.groupby("team")["minutes"].sum()

<StringArray>
['Jorge Luiz Frello Filho']
Length: 1, dtype: str [130]


team
Arsenal     850
Chelsea    1277
Name: minutes, dtype: int64